In [2]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('/content/global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))

Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         60  7168.0   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


Treemap: fossil fuel dependency by country

In [3]:
# Filter to fossil fuels only
fossil_df = df.loc[df['Source_Type'] == 'Fossil'].copy()

# Find most fossil-dependent region
top_region = (
    fossil_df.groupby('Region')['TWh']
    .sum()
    .sort_values(ascending=False)
    .idxmax()
)

# Treemap
fig = px.treemap(
    fossil_df,
    path=['Region', 'Country', 'Source'],
    values='TWh',
    color='Source',
    color_discrete_map={
        'Coal': '#4E79A7',         # blue
        'Oil': '#F28E2B',          # orange
        'Natural Gas': '#59A14F'   # green
    },
    title=f'{top_region} shows the highest fossil fuel dependence'
)

# Grey out parent nodes
fig.update_traces(
    root_color='lightgrey',
    textinfo='label+value'
)

fig.update_layout(
    margin=dict(t=60, l=10, r=10, b=10)
)

fig.show()

Sunburst: tipping behaviour by day and meal time

In [4]:
# Task 2 — Sunburst: tipping behaviour by day and meal time

# Load tips dataset
tips = px.data.tips()

# Aggregate total bill by hierarchy
tips_grouped = (
    tips.groupby(['day', 'time', 'smoker'])['total_bill']
    .sum()
    .reset_index()
)

# Identify highest spending group
top_day = (
    tips_grouped.groupby('day')['total_bill']
    .sum()
    .sort_values(ascending=False)
    .idxmax()
)

# Sunburst chart
fig = px.sunburst(
    tips_grouped,
    path=['day', 'time', 'smoker'],
    values='total_bill',
    color='smoker',
    color_discrete_map={
        'Yes': '#4E79A7',   # blue
        'No': '#F28E2B'     # orange
    },
    title=f'{top_day} has the highest total restaurant spending'
)

# Grey out parent nodes
fig.update_traces(
    root_color='lightgrey',
    textinfo='label+percent parent'
)

fig.update_layout(
    margin=dict(t=60, l=10, r=10, b=10)
)

fig.show()

Treemap vs bar: low-carbon energy by country

In [5]:
# Task 3 — Treemap vs bar: low-carbon energy by country

# Filter low-carbon sources
low_carbon_df = (
    df.loc[df['Source_Type'] == 'Low-carbon']
    .groupby('Country', as_index=False)['TWh']
    .sum()
)

# Add dummy root node
low_carbon_df['All'] = 'Low-carbon'

# Find leading country
top_country = (
    low_carbon_df.sort_values('TWh', ascending=False)
    .iloc[0]['Country']
)

# --- Treemap ---
treemap_fig = px.treemap(
    low_carbon_df,
    path=['All', 'Country'],
    values='TWh',
    color='TWh',
    color_continuous_scale='Blues',
    title='Low-carbon energy production by country'
)

treemap_fig.update_traces(
    root_color='lightgrey',
    textinfo='label+value'
)

treemap_fig.update_layout(
    margin=dict(t=60, l=10, r=10, b=10)
)

treemap_fig.show()

# --- Horizontal bar chart ---
bar_fig = px.bar(
    low_carbon_df.sort_values('TWh'),
    x='TWh',
    y='Country',
    orientation='h',
    color='TWh',
    color_continuous_scale='Blues',
    title=f'{top_country} leads low-carbon energy production'
)

bar_fig.update_traces(
    texttemplate='%{x:.0f}',
    textposition='outside'
)

bar_fig.update_layout(
    xaxis_title='TWh',
    yaxis_title='Country',
    showlegend=False
)

bar_fig.show()